Este projeto implementa um assistente de IA generativa utilizando Python e LLMs, com foco em construção de pipelines inteligentes e integração com dados externos. São aplicados conceitos de engenharia de prompts, uso de PromptTemplate/ChatPromptTemplate, geração de saídas estruturadas (JSON/stream) e técnicas de Retrieval-Augmented Generation (RAG) para consulta de documentos (.txt e .pdf).

Como caso de uso, foi desenvolvido um assistente para planejamento de viagens capaz de sugerir locais, analisar informações contextuais e responder de forma orientada a dados. A solução prioriza modularidade, escalabilidade e boas práticas no desenvolvimento de aplicações baseadas em IA

 LangChain = É um framework que ajuda a construir aplicações com IA generativa
 
 LLM = Modelo de linguagem usado é da OpenAI (Token)

In [0]:
### requirements.txt
%pip install \
  openai==1.86.0 \
  langchain==0.3.25 \
  langchain-openai==0.3.22 \
  langgraph==0.4.8 \
  python-dotenv==1.1.0 \
  faiss-cpu==1.11.0 \
  langchain-community==0.3.25 \
  pypdf==5.6.0


In [0]:
%pip install langchain-community

In [0]:
dbutils.library.restartPython()

In [0]:
OPENAI_API_KEY = ssss

In [0]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
import os 

api_key=OPENAI_API_KEY

numero_dias= 7
numero_criancas= 2
atividade= "natureza"

modelo_de_prompt = PromptTemplate(
   template = """
   Criar um roteiro de viagem de {numero_dias} dias,
    para uma familiar com {numero_criancas} criancas,
    que gostam de {atividade}
   """
)

prompt = modelo_de_prompt.format(
    numero_dias=numero_dias,
    numero_criancas=numero_criancas,
    atividade=atividade
)
print( "Prompt: \n", prompt)

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  #### nível de criatividade do modelo: quanto mais próximo de 2, mais criativo; quanto mais próximo de 0, menos criativo
    api_key=api_key
)
resposta = modelo.invoke(prompt)
print(resposta.content)


In [0]:
#### Orquestramento com cadeias LCEL no Lanchain = pipeline de execução de LLM
## 1- Modelo do prompt
## 2- Estrutura do LMM 
## 3- Saida de resposta 

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os 

api_key=OPENAI_API_KEY

modelo_cidade = PromptTemplate(
   template = """
   Sugira uma cidade dado meu interesse por {interesse}.
   """,
   input_variavels=["interesse"])

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

cadeia= modelo_cidade | modelo | StrOutputParser()

resposta = cadeia.invoke({"interesse": "praias"})
print(resposta)


##Esse código monta uma cadeia LCEL no LangChain que recebe um interesse do usuário, formata dinamicamente um prompt, envia para um modelo da OpenAI e retorna a resposta em texto limpo. É uma forma declarativa de orquestrar chamadas de LLM.

In [0]:
####Estruturando saidas com modelos um pouco mais complexo com Json 

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import Field, BaseModel ### biblioteca para estruturar saida
from langchain.globals import set_debug
import os 

api_key=OPENAI_API_KEY

###set_debug(True) utilizar para debugar cadeias

class Destino (BaseModel):
     cidade: str = Field( "Cidade recomendada para visitar")
     motivo: str = Field("Motivos para visitar essa cidade")
                    
parseador = JsonOutputParser(pydantic_object=Destino)


modelo_cidade = PromptTemplate(
   template = """
   Sugira uma cidade dado meu interesse por {interesse}.
   {formato_de_saida}
   """,
   input_variavels=["interesse"],
   partial_variables={"formato_de_saida": parseador.get_format_instructions()}
)

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

cadeia= modelo_cidade | modelo | parseador

resposta = cadeia.invoke({"interesse": "praias"})
print(resposta)


In [0]:

#### Sequencia de cadeias com LCEL

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import Field, BaseModel 
from langchain.globals import set_debug
import os 

api_key=OPENAI_API_KEY

class Destino (BaseModel):
     cidade: str = Field( "Cidade recomendada para visitar")
     motivo: str = Field("Motivos para visitar essa cidade")
                    
class Restaurantes (BaseModel):
     cidade: str = Field( "Cidade recomendada para visitar")
     restaurante: str = Field("Restaurantes recomendados na cidade")

parseador_destino = JsonOutputParser(pydantic_object=Destino)
parseador_restaurantes = JsonOutputParser(pydantic_object=Restaurantes)

modelo_cidade = PromptTemplate(
   template = """
   Sugira uma cidade dado meu interesse por {interesse}.
   {formato_de_saida}
   """,
   input_variavels=["interesse"],
   partial_variables={"formato_de_saida": parseador.get_format_instructions()}
)

modelo_restaurantes = PromptTemplate(
   template = """
   Sugira restaurantes populares entre locais {cidade}.
   {formato_de_saida}
   """,
   partial_variables={"formato_de_saida": parseador_restaurantes.get_format_instructions()}
)

modelo_cultural = PromptTemplate(
    template="""
    Sugira atividades e locais culturais em  {cidade}
    """,
)

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

### construo a cadeia de encadeamento da execucao
cadeia1= modelo_cidade | modelo | parseador
cadeia2= modelo_restaurantes | modelo | parseador_restaurantes
cadeia3= modelo_cultural | modelo | StrOutputParser()

cadeia = (cadeia1 | cadeia2 | cadeia3)

resposta = cadeia.invoke({"interesse": "praias"})
print(resposta)


In [0]:

#### Interacao de chat sem memoria

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from pydantic import Field, BaseModel 
from langchain.globals import set_debug
import os 

api_key=OPENAI_API_KEY

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

listas_perguntas= [
    "quero visitar um lugar no brasil, famoso por praias e cultura. Pode sugerir",
     "quero visitar lugares com bons drinks. Pode sugerir?",
     "quero visitar lugares com música ao vivo. Pode sugerir"
]

for uma_pergunta in listas_perguntas:
    resposta = modelo.invoke(uma_pergunta)
    print("Usuario:", uma_pergunta)
    print("Chatbot:", resposta.content, "\n")


    ### Inserir camada de gestão do historico para garantir a recuperação das informações
    ### LLM por padrao não mantém o contém das informações 

In [0]:

#### Adaptando o LCEL para dentro do historico

import os
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import InMemoryChatMessageHistory ## InMemoryChatMessageHistory é uma classe usada para armazenar o histórico de uma conversa entre usuário e modelo de IA durante a execução de um programa
from langchain_core.runnables.history import RunnableWithMessageHistory

api_key=OPENAI_API_KEY

modelo = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0.5,  
    api_key=api_key
)

prompt_sugestao = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de turismo. Apresente-se como Léo"),
    ("placeholder", "{historico}"),
    ("human","{query}")
]
)

cadeia= prompt_sugestao | modelo | StrOutputParser()

memoria = {}
sessao= 'memoria_modelo'


def historico_por_sessao(sessao : str):
    if sessao not in memoria:
        memoria[sessao] = InMemoryChatMessageHistory()
    return memoria[sessao]



lista_perguntas= [
    "quero visitar um lugar no brasil, famoso por praias e cultura. Pode sugerir",
     "quero visitar lugares com bons drinks. Pode sugerir?",
     "quero visitar lugares com música ao vivo. Pode sugerir"
]


cadeia_com_memoria = RunnableWithMessageHistory(
    runnable=cadeia,
    get_session_history=historico_por_sessao,
    input_messages_key="query",
    history_messages_key="historico"

)

### Inserir a cadeia de funcionamento de persitencia, trazendo a memoria para o modelo
for uma_pergunta in lista_perguntas:
    resposta = cadeia_com_memoria.invoke(
        {
            "query" : uma_pergunta
        },
        config={"session_id":sessao}
    )
    print("Usuário: ", uma_pergunta),
    print("IA: ", resposta, "\n")

###### ORQUESTRAMENTO COM LCEL E LANGGRAPH

LangGraph é uma ferramenta para construir sistemas de IA com vários agentes trabalhando juntos, controlando o fluxo de tarefas e mantendo memória do processo.

In [0]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict
import os

api_key=OPENAI_API_KEY

modelo = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.5,
    api_key=api_key
)


prompt_consultor_praias = ChatPromptTemplate.from_messages([
    ("system", "Você é um consultor de viagens. Apresente-se como Léo. Você é um especialista quando assunto é praias),
    ("human", "{query}")
]
)

prompt_consultor_montanha= ChatPromptTemplate.from_messages([
    ("system", "Você é um consultor de viagens. Apresente-se como Bru. Você é um especialista quando assunto é montanha e atividades radicais"),
    ("human", "{query}")
]
)

cadeia_consultor_praias = prompt_consultor_praias | modelo | StrOutputParser()
cadeia_consultor_montanha = prompt_consultor_montanha | modelo | StrOutputParser()


class Rota(TypedDict):
    destino: Literal["praia", "montanha"]

prompt_roteador = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda apenas com 'praia' ou 'montanha'"),
        ("human", "{query}")
    ]
)

roteador = propmt_roteador | modelo.with_structured_output(Rota)